# 02 Analysis - BahnDelayStory

Objective: turn the Phase 3 EDA into defensible descriptive findings about trend, train type, station, route, and hour effects.

This notebook intentionally avoids causal language. The right tool here is descriptive decomposition because the dataset has a known coverage break and no controls for weather, strikes, construction, or passenger volume.

Main analysis frame:
- use the stable 107-station panel for trend claims;
- use all stations for post-expansion outlier discovery;
- report cancellation separately from delay;
- rank contribution to late stops, not only late-share rates.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "src"))

from bahn_delay_story.config import PROCESSED_DIR  # noqa: E402
from bahn_delay_story.pipeline import duckdb_string_literal  # noqa: E402
from bahn_delay_story.quality import verify_database  # noqa: E402

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

TABLE_FILES = {
    "stops": "stops_clean.parquet",
    "station_day": "station_day_metrics.parquet",
    "train_day": "train_type_day_metrics.parquet",
    "hourly": "hourly_delay_metrics.parquet",
    "lines": "line_metrics.parquet",
}

missing = [name for name in TABLE_FILES.values() if not (PROCESSED_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing processed outputs: {missing}. Run `bahn-pipeline` first.")

con = duckdb.connect()
for view, file_name in TABLE_FILES.items():
    con.execute(
        f"CREATE OR REPLACE VIEW {view} AS "
        f"SELECT * FROM read_parquet({duckdb_string_literal(str(PROCESSED_DIR / file_name))})"
    )


def query(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()


def apply_layout(fig: go.Figure, height: int = 440) -> go.Figure:
    fig.update_layout(
        template="plotly_white",
        height=height,
        margin={"l": 40, "r": 25, "t": 70, "b": 45},
        legend_title_text="",
    )
    return fig


def percent_axis(fig: go.Figure, axis: str = "y") -> go.Figure:
    fig.update_layout(**{f"{axis}axis_tickformat": ".0%"})
    return fig

quality_report = verify_database()
quality_report["clean"]

## Stable-Station Panel

The coverage break makes all-station trend claims risky. The panel below keeps stations that are present in every month from January through October 2025. November and December remain visible for sensitivity, but the headline trend should be January-October.

In [ ]:
stable_panel = query("""
WITH stable AS (
  SELECT station_name
  FROM station_day
  WHERE station_name IS NOT NULL
    AND EXTRACT(MONTH FROM service_date) BETWEEN 1 AND 10
  GROUP BY station_name
  HAVING COUNT(DISTINCT EXTRACT(MONTH FROM service_date)) = 10
), monthly AS (
  SELECT
    EXTRACT(MONTH FROM sd.service_date) AS service_month,
    CASE WHEN sd.is_long_distance THEN 'Long-distance' ELSE 'Other services' END AS segment,
    SUM(stop_count) AS stop_count,
    SUM(stop_count * late_share_6_min) AS late_stops_6_min,
    SUM(stop_count * late_share_15_min) AS late_stops_15_min,
    SUM(stop_count * cancellation_share) AS canceled_stops,
    SUM(stop_count * avg_delay_min) AS delay_sum
  FROM station_day sd
  JOIN stable USING (station_name)
  GROUP BY 1, 2
)
SELECT
  service_month,
  segment,
  stop_count,
  late_stops_6_min / stop_count AS late_share_6_min,
  late_stops_15_min / stop_count AS late_share_15_min,
  canceled_stops / stop_count AS cancellation_share,
  delay_sum / stop_count AS avg_delay_min
FROM monthly
ORDER BY service_month, segment
""")
stable_panel.head()

In [ ]:
fig = px.line(
    stable_panel,
    x="service_month",
    y="late_share_6_min",
    color="segment",
    markers=True,
    labels={"service_month": "Month", "late_share_6_min": "Late share 6+ min", "segment": "Segment"},
    title="Stable-Station Late Share By Service Segment",
)
percent_axis(apply_layout(fig))
fig.show()

**Finding 1.** On the stable station panel, late share rises from January to October for both long-distance and other services. Long-distance remains structurally higher throughout the period.

In [ ]:
jan_oct_summary = query("""
WITH stable AS (
  SELECT station_name
  FROM station_day
  WHERE station_name IS NOT NULL
    AND EXTRACT(MONTH FROM service_date) BETWEEN 1 AND 10
  GROUP BY station_name
  HAVING COUNT(DISTINCT EXTRACT(MONTH FROM service_date)) = 10
), monthly AS (
  SELECT
    EXTRACT(MONTH FROM service_date) AS service_month,
    CASE WHEN is_long_distance THEN 'Long-distance' ELSE 'Other services' END AS segment,
    SUM(stop_count) AS stop_count,
    SUM(stop_count * late_share_6_min) AS late_stops_6_min,
    SUM(stop_count * cancellation_share) AS canceled_stops,
    SUM(stop_count * avg_delay_min) AS delay_sum
  FROM station_day
  JOIN stable USING (station_name)
  WHERE EXTRACT(MONTH FROM service_date) IN (1, 10)
  GROUP BY 1, 2
), wide AS (
  SELECT
    segment,
    SUM(CASE WHEN service_month = 1 THEN stop_count ELSE 0 END) AS jan_stops,
    SUM(CASE WHEN service_month = 1 THEN late_stops_6_min ELSE 0 END) AS jan_late,
    SUM(CASE WHEN service_month = 1 THEN canceled_stops ELSE 0 END) AS jan_canceled,
    SUM(CASE WHEN service_month = 1 THEN delay_sum ELSE 0 END) AS jan_delay_sum,
    SUM(CASE WHEN service_month = 10 THEN stop_count ELSE 0 END) AS oct_stops,
    SUM(CASE WHEN service_month = 10 THEN late_stops_6_min ELSE 0 END) AS oct_late,
    SUM(CASE WHEN service_month = 10 THEN canceled_stops ELSE 0 END) AS oct_canceled,
    SUM(CASE WHEN service_month = 10 THEN delay_sum ELSE 0 END) AS oct_delay_sum
  FROM monthly
  GROUP BY 1
)
SELECT
  segment,
  jan_stops,
  oct_stops,
  jan_late / jan_stops AS jan_late_share_6_min,
  oct_late / oct_stops AS oct_late_share_6_min,
  (oct_late / oct_stops) - (jan_late / jan_stops) AS late_share_change_pp,
  jan_canceled / jan_stops AS jan_cancellation_share,
  oct_canceled / oct_stops AS oct_cancellation_share,
  (oct_canceled / oct_stops) - (jan_canceled / jan_stops) AS cancellation_change_pp,
  jan_delay_sum / jan_stops AS jan_avg_delay_min,
  oct_delay_sum / oct_stops AS oct_avg_delay_min,
  (oct_delay_sum / oct_stops) - (jan_delay_sum / jan_stops) AS avg_delay_change_min,
  oct_late - jan_late AS late_stop_change
FROM wide
ORDER BY segment
""")
jan_oct_summary

In [ ]:
summary_long = jan_oct_summary.melt(
    id_vars=["segment"],
    value_vars=["late_share_change_pp", "cancellation_change_pp"],
    var_name="metric",
    value_name="share_point_change",
)
fig = px.bar(
    summary_long,
    x="metric",
    y="share_point_change",
    color="segment",
    barmode="group",
    text_auto=".1%",
    labels={"metric": "Metric", "share_point_change": "Jan-Oct change", "segment": "Segment"},
    title="Stable-Panel January To October Change",
)
percent_axis(apply_layout(fig))
fig.show()

**Finding 2.** The late-share increase is much larger than the cancellation-share change. The story should focus on delay deterioration, with cancellation as a separate secondary metric.

## Train-Type Decomposition

This section decomposes the January-to-October change in estimated late stops on the stable panel. It answers which train types explain the change, not just which train types have the highest late-share rate.

In [ ]:
train_change = query("""
WITH stable AS (
  SELECT station_name
  FROM station_day
  WHERE station_name IS NOT NULL
    AND EXTRACT(MONTH FROM service_date) BETWEEN 1 AND 10
  GROUP BY station_name
  HAVING COUNT(DISTINCT EXTRACT(MONTH FROM service_date)) = 10
), monthly AS (
  SELECT
    EXTRACT(MONTH FROM service_date) AS service_month,
    train_type,
    SUM(stop_count) AS stop_count,
    SUM(stop_count * late_share_6_min) AS late_stops
  FROM station_day
  JOIN stable USING (station_name)
  WHERE EXTRACT(MONTH FROM service_date) IN (1, 10)
  GROUP BY 1, 2
), wide AS (
  SELECT
    train_type,
    SUM(CASE WHEN service_month = 1 THEN stop_count ELSE 0 END) AS jan_stops,
    SUM(CASE WHEN service_month = 1 THEN late_stops ELSE 0 END) AS jan_late,
    SUM(CASE WHEN service_month = 10 THEN stop_count ELSE 0 END) AS oct_stops,
    SUM(CASE WHEN service_month = 10 THEN late_stops ELSE 0 END) AS oct_late
  FROM monthly
  GROUP BY 1
), changes AS (
  SELECT
    train_type,
    jan_stops,
    oct_stops,
    jan_late / NULLIF(jan_stops, 0) AS jan_late_share,
    oct_late / NULLIF(oct_stops, 0) AS oct_late_share,
    oct_late - jan_late AS late_stop_change
  FROM wide
  WHERE jan_stops >= 10000 OR oct_stops >= 10000
)
SELECT *
FROM changes
ORDER BY late_stop_change DESC
LIMIT 20
""")
train_change

In [ ]:
fig = px.bar(
    train_change.sort_values("late_stop_change"),
    x="late_stop_change",
    y="train_type",
    orientation="h",
    color="oct_late_share",
    color_continuous_scale="YlOrRd",
    labels={"late_stop_change": "Estimated late-stop change", "train_type": "Train type", "oct_late_share": "Oct late share"},
    title="Train Types Driving January-To-October Late-Stop Increase",
)
fig.update_layout(coloraxis_colorbar_tickformat=".0%")
apply_layout(fig, height=620)
fig.show()

**Finding 3.** The biggest contributors are high-volume RE, S, and RB services plus high-lateness ICE/IC services. The headline should avoid implying that only long-distance trains matter: local-service volume drives many late stops.

In [ ]:
fig = px.scatter(
    train_change,
    x="oct_late_share",
    y="late_stop_change",
    size="oct_stops",
    color="train_type",
    hover_name="train_type",
    labels={"oct_late_share": "October late share", "late_stop_change": "Jan-Oct late-stop change", "oct_stops": "October stops"},
    title="Rate Versus Contribution By Train Type",
)
percent_axis(apply_layout(fig), "x")
fig.show()

**Finding 4.** Rate and contribution are different. ICE has a very high October late share, but RE/S/RB create large late-stop changes because they have far more stops.

## Station Decomposition

This section identifies where the increase concentrates. It uses contribution to added late stops rather than only ranking by late-share rate.

In [ ]:
station_change = query("""
WITH stable AS (
  SELECT station_name
  FROM station_day
  WHERE station_name IS NOT NULL
    AND EXTRACT(MONTH FROM service_date) BETWEEN 1 AND 10
  GROUP BY station_name
  HAVING COUNT(DISTINCT EXTRACT(MONTH FROM service_date)) = 10
), monthly AS (
  SELECT
    EXTRACT(MONTH FROM service_date) AS service_month,
    station_name,
    SUM(stop_count) AS stop_count,
    SUM(stop_count * late_share_6_min) AS late_stops
  FROM station_day
  JOIN stable USING (station_name)
  WHERE EXTRACT(MONTH FROM service_date) IN (1, 10)
  GROUP BY 1, 2
), wide AS (
  SELECT
    station_name,
    SUM(CASE WHEN service_month = 1 THEN stop_count ELSE 0 END) AS jan_stops,
    SUM(CASE WHEN service_month = 1 THEN late_stops ELSE 0 END) AS jan_late,
    SUM(CASE WHEN service_month = 10 THEN stop_count ELSE 0 END) AS oct_stops,
    SUM(CASE WHEN service_month = 10 THEN late_stops ELSE 0 END) AS oct_late
  FROM monthly
  GROUP BY 1
)
SELECT
  station_name,
  jan_stops,
  oct_stops,
  jan_late / jan_stops AS jan_late_share,
  oct_late / oct_stops AS oct_late_share,
  oct_late - jan_late AS late_stop_change
FROM wide
ORDER BY late_stop_change DESC
LIMIT 25
""")
station_change.head(20)

In [ ]:
fig = px.bar(
    station_change.head(20).sort_values("late_stop_change"),
    x="late_stop_change",
    y="station_name",
    orientation="h",
    color="oct_late_share",
    color_continuous_scale="YlOrRd",
    labels={"late_stop_change": "Estimated late-stop change", "station_name": "Station", "oct_late_share": "Oct late share"},
    title="Stations Driving January-To-October Late-Stop Increase",
)
fig.update_layout(coloraxis_colorbar_tickformat=".0%")
apply_layout(fig, height=650)
fig.show()

**Finding 5.** Added late stops concentrate at major hubs such as Stuttgart, München, Hamburg, Köln, Berlin, Dortmund, and Hannover. This is a practical dashboard dimension: users need station contribution and rate views.

## Hour Decomposition

This section checks whether the added late stops concentrate in specific hours. It uses raw `stops_clean` for the stable stations so the hour-level change is exact for the panel.

In [ ]:
hour_change = query("""
WITH stable AS (
  SELECT station_name
  FROM station_day
  WHERE station_name IS NOT NULL
    AND EXTRACT(MONTH FROM service_date) BETWEEN 1 AND 10
  GROUP BY station_name
  HAVING COUNT(DISTINCT EXTRACT(MONTH FROM service_date)) = 10
), stable_stops AS (
  SELECT s.*
  FROM stops s
  JOIN stable USING (station_name)
  WHERE service_month IN (1, 10)
), monthly AS (
  SELECT
    service_month,
    hour_of_day,
    COUNT(*) AS stop_count,
    SUM(CASE WHEN is_late_6_min THEN 1 ELSE 0 END) AS late_stops
  FROM stable_stops
  GROUP BY 1, 2
), wide AS (
  SELECT
    hour_of_day,
    SUM(CASE WHEN service_month = 1 THEN stop_count ELSE 0 END) AS jan_stops,
    SUM(CASE WHEN service_month = 1 THEN late_stops ELSE 0 END) AS jan_late,
    SUM(CASE WHEN service_month = 10 THEN stop_count ELSE 0 END) AS oct_stops,
    SUM(CASE WHEN service_month = 10 THEN late_stops ELSE 0 END) AS oct_late
  FROM monthly
  GROUP BY 1
)
SELECT
  hour_of_day,
  jan_stops,
  oct_stops,
  jan_late / jan_stops AS jan_late_share,
  oct_late / oct_stops AS oct_late_share,
  oct_late - jan_late AS late_stop_change
FROM wide
ORDER BY hour_of_day
""")
hour_change.head()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=hour_change["hour_of_day"], y=hour_change["late_stop_change"], name="Late-stop change"))
fig.add_trace(go.Scatter(x=hour_change["hour_of_day"], y=hour_change["oct_late_share"], name="October late share", yaxis="y2", mode="lines+markers"))
fig.update_layout(
    title="Hour Contribution To January-To-October Late-Stop Increase",
    xaxis_title="Hour of day",
    yaxis_title="Estimated late-stop change",
    yaxis2={"title": "October late share", "overlaying": "y", "side": "right", "tickformat": ".0%"},
)
apply_layout(fig)
fig.show()

**Finding 6.** The largest added late-stop contribution is in the afternoon and evening, especially 14:00-21:00. That supports the time-window hypothesis without relying on low-volume overnight cells.

## Route/Service Sensitivity

Line rankings are useful for the dashboard, but they are sensitive to minimum-stop thresholds. This section quantifies that sensitivity before choosing headline route examples.

In [ ]:
line_floor = query("""
SELECT
  floor_value,
  COUNT(*) AS services,
  AVG(late_share_6_min) AS avg_late_share,
  MAX(late_share_6_min) AS max_late_share
FROM (
  SELECT 100 AS floor_value, * FROM lines WHERE stop_count >= 100
  UNION ALL SELECT 500 AS floor_value, * FROM lines WHERE stop_count >= 500
  UNION ALL SELECT 1000 AS floor_value, * FROM lines WHERE stop_count >= 1000
  UNION ALL SELECT 5000 AS floor_value, * FROM lines WHERE stop_count >= 5000
)
GROUP BY 1
ORDER BY 1
""")
line_floor

In [ ]:
fig = px.line(
    line_floor,
    x="floor_value",
    y="max_late_share",
    markers=True,
    labels={"floor_value": "Minimum stops", "max_late_share": "Max late share"},
    title="Route Ranking Sensitivity To Stop-Count Floor",
)
percent_axis(apply_layout(fig))
fig.show()

**Finding 7.** Route outlier claims need a floor. The maximum late share falls once the minimum-stop floor reaches 5,000, so the final story should avoid ranking very small services as if they were system-level patterns.

In [ ]:
route_outliers = query("""
SELECT
  train_type,
  train_name,
  final_destination_station,
  stop_count,
  late_share_6_min,
  avg_delay_min,
  p90_delay_min,
  cancellation_share
FROM lines
WHERE stop_count >= 1000
ORDER BY late_share_6_min DESC, stop_count DESC
LIMIT 20
""")
route_outliers["service"] = route_outliers["train_name"] + " -> " + route_outliers["final_destination_station"].fillna("unknown")
route_outliers.head(10)

In [ ]:
fig = px.bar(
    route_outliers.sort_values("late_share_6_min"),
    x="late_share_6_min",
    y="service",
    orientation="h",
    color="train_type",
    labels={"late_share_6_min": "Late share 6+ min", "service": "Service"},
    title="High-Late Services With At Least 1,000 Stops",
)
percent_axis(apply_layout(fig, height=650), "x")
fig.show()

**Finding 8.** Even with a 1,000-stop floor, the highest-late services are dominated by long-distance trains. Route examples can support the story, but they should be presented as examples, not as causal explanations.

## Phase 4 Conclusions

1. **Headline direction:** on the stable 107-station panel, delays are materially worse by October than January 2025.
2. **Most defensible headline metric:** late share 6+ minutes, with p90 delay as the severe-tail companion metric.
3. **Main split:** long-distance versus other services. Long-distance trains are much later, but local services contribute many added late stops because of volume.
4. **Main decomposition:** RE, S, RB, ICE, and IC account for the largest added late-stop contribution from January to October.
5. **Where:** added late stops concentrate at major hubs including Stuttgart, München, Hamburg, Köln, Berlin, Dortmund, and Hannover.
6. **When:** afternoon/evening windows drive the biggest added late-stop contribution.
7. **Scope:** the final story is a 2025 stable-panel analysis. Headline trend claims compare January and October; November/December are excluded from the trend headline because of the coverage expansion.

Recommended dashboard views:
- stable-station monthly trend by service segment;
- train-type contribution ranking;
- station contribution ranking;
- hour-of-day contribution view;
- route/service outliers with configurable minimum-stop floor.